# DeepMatch 样例代码
- https://github.com/shenweichen/DeepMatch
- https://deepmatch.readthedocs.io/en/latest/

# 下载movielens-1M数据 安装依赖包

In [ ]:
#! wget http://files.grouplens.org/datasets/movielens/ml-1m.zip -O ./ml-1m.zip 
#! wget https://raw.githubusercontent.com/shenweichen/DeepMatch/master/examples/preprocess.py -O preprocess.py
#! unzip -o ml-1m.zip 
#! pip uninstall -y -q tensorflow
#! pip install -q tensorflow-gpu==2.5.0
#! pip install -q deepmatch

--2022-07-02 18:35:40--  http://files.grouplens.org/datasets/movielens/ml-1m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.65.152
Connecting to files.grouplens.org (files.grouplens.org)|128.101.65.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5917549 (5.6M) [application/zip]
Saving to: ‘./ml-1m.zip’

./ml-1m.zip         100%[===================>]   5.64M  3.43MB/s    in 1.6s    

2022-07-02 18:35:42 (3.43 MB/s) - ‘./ml-1m.zip’ saved [5917549/5917549]

--2022-07-02 18:35:42--  https://raw.githubusercontent.com/shenweichen/DeepMatch/master/examples/preprocess.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5642 (5.5K) [text/plain]
Saving to: ‘preprocess.py’

preprocess.py       100%[===================>]  

# 导入需要的库

In [1]:
import pandas as pd
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler

# 这一行负责从 preprocess.py 文件中加载 gen_model_input 函数
from preprocess import gen_data_set, gen_model_input 

# 读取数据

In [2]:
data_path = "./"

unames = ['user_id','gender','age','occupation','zip']
user = pd.read_csv(data_path+'ml-1m/users.dat',sep='::',header=None,names=unames)
rnames = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(data_path+'ml-1m/ratings.dat',sep='::',header=None,names=rnames)
mnames = ['movie_id','title','genres']
movies = pd.read_csv(data_path+'ml-1m/movies.dat',sep='::',header=None,names=mnames,encoding="unicode_escape")
movies['genres'] = list(map(lambda x: x.split('|')[0], movies['genres'].values))

#pd.merge 函数在不指定 on 参数时，会自动寻找两个表中名称相同的列作为连接键（Key）。
# 默认的合并方式是 Inner Join（内连接），即只保留两个表中都存在的 Key。
data = pd.merge(pd.merge(ratings,movies),user)#.iloc[:10000]
#例如Pandas 发现ratings表和movies表都有 movie_id 这一列，于是自动将其作为连接键。

C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_22784\4024406526.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  user = pd.read_csv(data_path+'ml-1m/users.dat',sep='::',header=None,names=unames)
C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_22784\4024406526.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_csv(data_path+'ml-1m/ratings.dat',sep='::',header=None,names=rnames)
C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_22784\4024406526.py:8: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 cha

In [3]:
user

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [4]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [5]:
movies

,movie_id,title,genres
0,1,Toy Story (1995),Animation
1,2,Jumanji (1995),Adventure
2,3,Grumpier Old Men (1995),Comedy
3,4,Waiting to Exhale (1995),Comedy
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [6]:
data

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,978302109,James and the Giant Peach (1996),Animation,F,1,10,48067
2,1,914,3,978301968,My Fair Lady (1964),Musical,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation,F,1,10,48067
...,...,...,...,...,...,...,...,...,...,...
1000204,4211,3791,2,965319075,Footloose (1984),Drama,M,45,5,77662
1000205,4211,3806,3,965319138,MacKenna's Gold (1969),Western,M,45,5,77662
1000206,4211,3840,4,965319197,Pumpkinhead (1988),Horror,M,45,5,77662
1000207,4211,3766,2,965319138,Missing in Action (1984),Action,M,45,5,77662


# 构建特征列，训练模型，导出embedding

In [3]:
#data = pd.read_csvdata = pd.read_csv("./movielens_sample.txt")
sparse_features = ["movie_id", "user_id",
                    "gender", "age", "occupation", "zip", "genres"]
SEQ_LEN = 50
negsample = 0

In [4]:
# 1.Label Encoding for sparse features,and process sequence features with `gen_date_set` and `gen_model_input`

#features = ['user_id','movie_id','gender', 'age', 'occupation', 'zip']
feature_max_idx = {}
for feature in sparse_features:
    # 实例化 LabelEncoder（标签编辑器），将离散标签（字符串或者数字）转化成连续的整数
    lbe = LabelEncoder()
    ## 核心动作：String -> Int，并且 +1
    data[feature] = lbe.fit_transform(data[feature]) + 1
    # 记录最大值，用于定义 Embedding 矩阵的大小
    feature_max_idx[feature] = data[feature].max() + 1


#原始的 data 表是合并后的交互记录，同一个 User 对应多行数据。每一行是一次交互的评分，一个User会有很多行，但是性别、年龄等静态属性是重复的
#这里提取出唯一的 User ID -> 用户属性 和 Movie ID -> 物品属性 的映射关系。

#按照user_id去重，保留性别、年龄、职业、邮编等用户属性，提取出一张用户花名册（静态属性）
user_profile = data[["user_id", "gender", "age", "occupation", "zip"]].drop_duplicates('user_id')
#按照movie_id去重，保留电影类别等物品属性，提取出一张电影信息表
item_profile = data[["movie_id","genres"]].drop_duplicates('movie_id')


print('构建去重画像 (Profile Extraction)')
print(user_profile.shape)
print(user_profile.head())
print(item_profile.shape)
print(item_profile.head())
print('=======================================')

构建去重画像 (Profile Extraction)
(6040, 5)
     user_id  gender  age  occupation   zip
0          1       1    1          11  1589
53         2       2    7          17  2249
182       12       2    3          13  1166
205       15       2    3           8   905
406       17       2    6           2  3188
(3706, 2)
   movie_id  genres
0      1105       8
1       640       3
2       854      12
3      3178       8
4      2163       3


In [5]:
#把 user_id 设为 user_profile 这个 DataFrame 的索引（Index）。
user_profile.set_index("user_id", inplace=True)
print('设置用户ID为索引 (Set User ID as Index)')
print(user_profile.head())
print('=======================================')

#构建用户的原始历史行为序列
user_item_list = data.groupby("user_id")['movie_id'].apply(list)
print('构建用户-物品交互列表 (User-Item Interaction List)')
print(user_item_list.head())
print('=======================================')

设置用户ID为索引 (Set User ID as Index)
         gender  age  occupation   zip
user_id                               
1             1    1          11  1589
2             2    7          17  2249
12            2    3          13  1166
15            2    3           8   905
17            2    6           2  3188
构建用户-物品交互列表 (User-Item Interaction List)
user_id
1    [1105, 640, 854, 3178, 2163, 1108, 1196, 2600,...
2    [1105, 2890, 2129, 1783, 1118, 1849, 1155, 126...
3    [2163, 1108, 1179, 1782, 254, 2899, 628, 1121,...
4    [1026, 2489, 254, 1849, 3236, 1121, 467, 1107,...
5    [3178, 2163, 859, 2890, 1575, 2558, 145, 2489,...
Name: movie_id, dtype: object


In [10]:
#数据集切分#来自preprocess.py
#SEQ_LEN = 50 序列最大长度。如果用户看了超过50个电影，只截取最近的50个。
#negsample = 0 训练时是否进行“显式”负采样。这里设为0，因为后续用了 In-Batch Softmax（即隐式负采样）。
train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)  
print('数据集切分 (Dataset Splitting)')
print(len(train_set), len(test_set))
print(train_set[:5])
print(test_set[:5])
print('=======================================')

100%|██████████| 6040/6040 [00:04<00:00, 1442.84it/s]



8 8
数据集切分 (Dataset Splitting)
988129 6040
[(1675, 2130, 1, [318, 3127, 2172, 1854, 1421, 1911, 1615, 180, 3456, 797, 250, 3457, 1293, 973, 3398, 65, 2819, 653, 24, 738, 2336, 308, 2414, 1697, 2820, 3528, 2201, 1874, 2120, 1833, 3075, 1914, 3046, 2412, 2460, 746, 2440, 75, 2185, 2334, 429, 2338, 2333, 1473, 320, 596, 2495, 3460, 2339, 1913], 50, [1, 15, 1, 15, 1, 4, 2, 15, 1, 1, 5, 1, 1, 1, 1, 15, 15, 2, 8, 1, 1, 1, 2, 1, 5, 2, 1, 2, 13, 5, 1, 1, 2, 1, 15, 5, 1, 15, 11, 1, 1, 1, 1, 15, 1, 1, 1, 6, 1, 2], 1, 2), (2436, 1900, 1, [2058, 3352, 51, 339, 1929, 3234, 1704, 3406, 1084, 2991, 3294, 1365, 2271, 1505, 1964, 2839, 3430, 2708, 3190, 2163, 1174, 3193, 1216, 3166, 34, 1, 1194, 1145, 1148, 2401, 1201, 381, 2817, 2600, 1067, 1029, 1632, 746, 3, 344, 992, 679, 800, 1614, 427, 584, 3375, 2303, 348, 3130], 50, [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 4, 4, 3, 5, 5, 5, 5, 5, 1, 5, 5, 3, 1, 1, 5, 5, 1, 8, 5, 5, 5, 5, 5, 5, 5, 5, 5], 3, 2), (1584, 2167, 1, [1501, 

In [11]:
# 生成模型输入#来自preprocess.py
train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)
print('生成模型输入 (Generate Model Input)')

# 这是一个字典，不能直接用 train_model_input[:5] 切片
# 我们遍历字典，打印每个特征的前2条数据来看看
print("\n--- 训练集输入样例 (Train Input Sample) ---")
for key in ['user_id', 'hist_movie_id']: # 只看几个关键特征，避免刷屏
    if key in train_model_input:
        print(f"Feature: {key}")
        print(train_model_input[key][:2]) 

print("\n--- 训练集标签样例 (Train Label Sample) ---")
print(train_label[:5])

print("\n--- 测试集标签样例 (Test Label Sample) ---")
print(test_label[:5])
print('=======================================')

生成模型输入 (Generate Model Input)

--- 训练集输入样例 (Train Input Sample) ---
Feature: user_id
[1675 2436]
Feature: hist_movie_id
[[ 318 3127 2172 1854 1421 1911 1615  180 3456  797  250 3457 1293  973
  3398   65 2819  653   24  738 2336  308 2414 1697 2820 3528 2201 1874
  2120 1833 3075 1914 3046 2412 2460  746 2440   75 2185 2334  429 2338
  2333 1473  320  596 2495 3460 2339 1913]
 [2058 3352   51  339 1929 3234 1704 3406 1084 2991 3294 1365 2271 1505
  1964 2839 3430 2708 3190 2163 1174 3193 1216 3166   34    1 1194 1145
  1148 2401 1201  381 2817 2600 1067 1029 1632  746    3  344  992  679
   800 1614  427  584 3375 2303  348 3130]]

--- 训练集标签样例 (Train Label Sample) ---
[1 1 1 1 1]

--- 测试集标签样例 (Test Label Sample) ---
[1 1 1 1 1]


In [12]:
# 2.count #unique features for each sparse field and generate feature config for sequence feature

embedding_dim = 32  #embedding的维度

#构建用户塔特征
user_feature_columns = [
                        ## 1). 稀疏特征 (Categorical Features)
                        #feature_column.py中
                        SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        #SparseFeat：这是 DeepCTR/DeepMatch 库定义的一个类，专门处理 类别型/离散型特征。
                        #这是特征内部的 Embedding 维度。比如“性别”只映射成 16 维，“年龄”只映射成 16 维。这些向量后续会在神经网络里拼接（Concat）起来，经过全连接层（MLP），最终才变成 32 维。

                        ## 2). 变长序列特征 (Sequence Features)
                        #处理用户历史行为序列的关键
                        # #核心机制 (Shared Embedding): 
                        # #注意 embedding_name="movie_id"。
                        # #这意味着用户历史记录里的电影 ID，和物品侧的电影 ID，共享同一张 Embedding 查找表。
                        # #这是推荐系统常用的技巧
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]
                        #combiner (默认 "mean")
                        #含义: 池化策略（Pooling Strategy）。
                        #问题: 查表后，一个用户变成了 50×32的矩阵（50个行为，每个32维）。但后面的全连接层（Dense Layer）通常需要一个扁平的向量。
                        #解决：mean\sum\max
'''
train_model_input['hist_movie_id']，是一个 shape 为 (Batch, 50) 的整数矩阵。
查表: 拿着这些整数，去查那个叫 "movie_id" 的 Embedding 表（这张表定义在里面那个 SparseFeat 里）。把 50 个 ID 变成 50 个向量。
处理变长: 虽然矩阵是 50 宽，但我知道有些后面是补 0 的。去 train_model_input['hist_len'] 看看真实长度是多少。
合并: 把这 50 个向量里的前 hist_len 个有效向量加起来，除以 hist_len（因为 combiner='mean'）。
输出: 最终得到一个 (Batch, 32) 的向量，作为一个普通的特征输入到后面的 MLP 层中。
'''


#3). 构建物品塔特征 (Item Feature Columns)
item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
                        SparseFeat('genres', feature_max_idx['genres'], embedding_dim)
                       ]

#user_feature_columns/item_feature_columns一个包含多个配置对象（Config Objects）的 Python 列表（List）。
#它们是整个 DeepMatch/DeepCTR 框架的 “施工图纸”。它们不包含真实的数据（数据在 x_train 里），它们只包含数据的元数据（Metadata），告诉模型该如何处理每一列数据。
'''
它包含了什么信息？
每个元素（SparseFeat 或 VarLenSparseFeat）都是一个从 namedtuple 派生出来的对象，这使得它们非常紧凑。

你是谁？(name): "我是 'user_id' 列"。这告诉 build_input_features 创建 Input 层时，名字要叫 "user_id"，也告诉 input_from_feature_columns 去 features 字典里找 "user_id" 这个 key。
你多大？(vocabulary_size): "这列最大可能有 6040 个不同的 ID"。这告诉 create_embedding_matrix 要申请一个多大的矩阵（6040行）。
你要变成多宽？(embedding_dim): "把每个 ID 变成一个长度为 4 的向量"。这决定了 Embedding 矩阵的列数。
你是哪一伙的？(embedding_name vs name):
        SparseFeat('hist_movie_id', embedding_name='movie_id')
        这告诉模型："虽然我的数据叫 'hist_movie_id'，但我其实是电影，请让我去查 'movie_id' 那个表，别给我单开小灶。" 这是实现共享 Embedding 的关键配置。
你有什么特殊癖好？:
        VarLenSparseFeat 里的 maxlen=50：告诉模型这列数据是个序列，最长50。
        combiner='mean'：告诉模型查完表后，把这50个向量取个平均值。
'''

'\n它包含了什么信息？\n每个元素（SparseFeat 或 VarLenSparseFeat）都是一个从 namedtuple 派生出来的对象，这使得它们非常紧凑。\n\n你是谁？(name): "我是 \'user_id\' 列"。这告诉 build_input_features 创建 Input 层时，名字要叫 "user_id"，也告诉 input_from_feature_columns 去 features 字典里找 "user_id" 这个 key。\n你多大？(vocabulary_size): "这列最大可能有 6040 个不同的 ID"。这告诉 create_embedding_matrix 要申请一个多大的矩阵（6040行）。\n你要变成多宽？(embedding_dim): "把每个 ID 变成一个长度为 4 的向量"。这决定了 Embedding 矩阵的列数。\n你是哪一伙的？(embedding_name vs name):\n        SparseFeat(\'hist_movie_id\', embedding_name=\'movie_id\')\n        这告诉模型："虽然我的数据叫 \'hist_movie_id\'，但我其实是电影，请让我去查 \'movie_id\' 那个表，别给我单开小灶。" 这是实现共享 Embedding 的关键配置。\n你有什么特殊癖好？:\n        VarLenSparseFeat 里的 maxlen=50：告诉模型这列数据是个序列，最长50。\n        combiner=\'mean\'：告诉模型查完表后，把这50个向量取个平均值。\n'

In [13]:
user_feature_columns

[SparseFeat(name='user_id', vocabulary_size=6041, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x0000017D9EDC71F0>, embedding_name='user_id', group_name='default_group', trainable=True),
 SparseFeat(name='gender', vocabulary_size=3, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x0000017D3DFDE520>, embedding_name='gender', group_name='default_group', trainable=True),
 SparseFeat(name='age', vocabulary_size=8, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x0000017D3DFDEAF0>, embedding_name='age', group_name='default_group', trainable=True),
 SparseFeat(name='occupation', vocabulary_size=22, embedding_dim=16, use_hash

In [14]:
item_feature_columns

[SparseFeat(name='movie_id', vocabulary_size=3707, embedding_dim=32, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x0000017D3EB8A820>, embedding_name='movie_id', group_name='default_group', trainable=True),
 SparseFeat(name='genres', vocabulary_size=19, embedding_dim=32, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x0000017D3EB8A8B0>, embedding_name='genres', group_name='default_group', trainable=True)]

In [15]:
#4). 负采样配置 (Negative Sampler Config)
from collections import Counter

##统计训练集中每个电影出现的次数
train_counter = Counter(train_model_input['movie_id'])
#结果：{1: 500次, 2: 12次, 3: 10000次...}。这就是所谓的“频次直方图”。

##生成一个列表，索引是 movie_id，值是该 movie_id 出现的次数
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
#转化成数组：模型（TensorFlow）不认识 Python 字典，它只认数组。
#逻辑：
#按照 movie_id 从 0 到 Max 的顺序，把频次排成一个长长的列表。
#get(i, 0)：如果某个 ID 没出现过，填 0。
#item_feature_columns[0].vocabulary_size：确保列表长度覆盖了所有可能的电影 ID。


'''
为什么要统计 item_count?
在进行采样 Softmax (Sampled Softmax) 时，
我们通常希望对热门物品进行降权，或者在计算概率时修正偏差（bias）。
item_count 提供了热门程度的信息，防止极其热门的物品主导了梯度的更新。
'''
#s(u,i)=ModelScore(u,i)−log(Prob(i))
#Prob(i) 是物品 i 出现的概率（频率）。
#−log(Prob(i)) 这一项的作用是：如果一个物品特别热门（概率大），我就要减掉一部分分非数。 这强迫模型去学习用户和物品真正的匹配度，排除掉“因为它热门所以你点了”的干扰因素。
#DeepMatch 的 sampledsoftmaxloss 内部实现了这个逻辑，而 item_count 就是用来计算这个 Prob(𝑖)的原材料。

'\n为什么要统计 item_count?\n在进行采样 Softmax (Sampled Softmax) 时，\n我们通常希望对热门物品进行降权，或者在计算概率时修正偏差（bias）。\nitem_count 提供了热门程度的信息，防止极其热门的物品主导了梯度的更新。\n'

In [16]:
train_counter

Counter({2652: 3397,
         254: 2880,
         1107: 2877,
         576: 2625,
         1121: 2619,
         467: 2572,
         2375: 2570,
         1849: 2554,
         1450: 2513,
         580: 2497,
         594: 2485,
         1179: 2455,
         2558: 2431,
         1109: 2398,
         107: 2365,
         2204: 2350,
         1174: 2258,
         1108: 2239,
         2786: 2227,
         310: 2212,
         514: 2208,
         1486: 2199,
         1026: 2176,
         2512: 2170,
         288: 2156,
         2427: 2148,
         803: 2130,
         1149: 2082,
         347: 2069,
         1: 2064,
         1125: 2008,
         2709: 1988,
         444: 1970,
         3342: 1885,
         1111: 1808,
         528: 1783,
         50: 1773,
         1168: 1770,
         34: 1742,
         2776: 1736,
         2959: 1718,
         738: 1711,
         2587: 1709,
         859: 1707,
         1051: 1707,
         864: 1698,
         2163: 1690,
         1289: 1688,
         972: 1

In [17]:
item_count

[0,
 2064,
 697,
 472,
 170,
 291,
 931,
 457,
 67,
 102,
 886,
 1023,
 158,
 96,
 153,
 143,
 677,
 832,
 156,
 382,
 157,
 1343,
 377,
 124,
 623,
 974,
 98,
 60,
 176,
 400,
 74,
 138,
 1502,
 4,
 1742,
 68,
 918,
 8,
 27,
 1353,
 30,
 240,
 220,
 165,
 311,
 542,
 165,
 1125,
 376,
 27,
 1773,
 429,
 8,
 39,
 45,
 9,
 97,
 499,
 8,
 351,
 66,
 537,
 110,
 74,
 141,
 97,
 4,
 60,
 316,
 910,
 93,
 93,
 221,
 120,
 11,
 178,
 34,
 50,
 113,
 52,
 164,
 88,
 33,
 20,
 192,
 225,
 76,
 187,
 228,
 6,
 87,
 89,
 178,
 631,
 13,
 44,
 6,
 49,
 128,
 250,
 59,
 32,
 676,
 383,
 12,
 252,
 7,
 2365,
 1195,
 565,
 52,
 11,
 40,
 78,
 66,
 13,
 3,
 37,
 192,
 114,
 14,
 316,
 99,
 1,
 12,
 13,
 2,
 16,
 89,
 1,
 2,
 129,
 8,
 12,
 2,
 1,
 139,
 675,
 1,
 426,
 359,
 18,
 181,
 23,
 47,
 1241,
 550,
 69,
 772,
 166,
 87,
 66,
 185,
 254,
 116,
 557,
 753,
 487,
 537,
 377,
 823,
 60,
 5,
 360,
 109,
 315,
 92,
 456,
 558,
 81,
 217,
 205,
 139,
 18,
 48,
 449,
 91,
 49,
 29,
 26,
 563,
 205,


In [18]:
## 定义采样器
sampler_config = NegativeSampler('inbatch',num_sampled=255,item_name="movie_id",item_count=item_count)
'''
'inbatch': 这是一个非常高效的训练策略。
    传统做法：
        对于每个 User，还要单独随机采样几千个他没看过的 Item 作为负样本。这会产生巨大的 IO 和计算开销。
    In-Batch 做法：
        假设一个 Batch 有 256 条数据 (User A - Movie A, User B - Movie B, ...)。
        对于 User A 来说，除了他自己看过的 Movie A 是正样本，
        Batch 里其他 255 个 User 看过的电影（Movie B, Movie C...）都可以天然作为负样本。
'''
'''
num_sampled=255: 在 Batch 内部采样多少个负样本。配合后面的 batch_size=256 使用。
'''

'\nnum_sampled=255: 在 Batch 内部采样多少个负样本。配合后面的 batch_size=256 使用。\n'

In [19]:
# 3.Define Model and train
#定义模型与训练 (Define Model and Train)
#这一阶段的目标是让模型学习如何将 User 和 Item 映射到同一个向量空间，使得用户和他感兴趣的物品距离更近。

import tensorflow as tf
#禁用了 TensorFlow 2.x 默认的 Eager Execution（动态图模式），强制使用 Graph Execution（静态图模式）。
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
'''
原因：
In-Batch Softmax 的特殊性：
    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。
    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，
    这对于大规模分类和负采样训练至关重要。
库的兼容性：
    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，
    关闭 Eager 模式能保证最大程度的兼容和稳定。
'''

'\n原因：\nIn-Batch Softmax 的特殊性：\n    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。\n    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，\n    这对于大规模分类和负采样训练至关重要。\n库的兼容性：\n    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，\n    关闭 Eager 模式能保证最大程度的兼容和稳定。\n'

##### 至此完成了准备数据（preprocess）、配置特征列（SparseFeat/VarLenSparseFeat）、配置负采样策略（NegativeSampler

#### 开始以构建并训练DSSM

In [20]:
#初始化 DSSM 模型
model = DSSM(user_feature_columns, item_feature_columns,user_dnn_hidden_units=(128,64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,),loss_type='softmax',sampler_config=sampler_config)
#关键参数 loss_type='softmax'：告诉模型我们不是在其做简单的二分类（点击/不点击），
# #而是一个多分类问题（从海量物品中找出用户最可能点击的那个）。
# #配合之前定义的 sampler_config，模型会使用高效的 In-Batch 采样来计算 Loss。

#编译（compile）与拟合
#这一步是在配置训练过程。此时虽然模型结构（双塔网络）已经搭好了，但它还不知道“怎么学（优化）”以及“学得好不好（损失）”。
model.compile(optimizer="adam", loss=sampledsoftmaxloss)
#sampledsoftmaxloss：
#这里的猫腻: 你可能会疑惑，我们在 DSSM 的内部代码 InBatchSoftmaxLayer 里不是已经算过 Loss 了吗？为什么这里还要再传一个 Loss？
#真相: 这是一个假的 Loss 函数（Dummy Loss）。



##！！！！开始训练
history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=256, epochs=20, verbose=1, validation_split=0.0, )
####batch_size=256：这意味着如果不使用额外的负采样，每个正样本天然对应了 255 个负样本。
#train_model_input (输入材料):这是一个字典 {'user_id': [...], 'movie_id': [...], ...}。Keras 会根据 key 把数据分发给模型里对应的 Input 层插座。
#train_label (标准答案):在普通的二分类（比如 deepctr）里，这里应该是 0 或 1。在 In-Batch Softmax 里: 这个 label 其实并没有被真正的 Loss 计算逻辑用到！
    #回想一下 InBatchSoftmaxLayer 的代码：它只用了 inputs 里的 ID，并没有用这里的 y_true。
    #那这里为什么要传？因为 Keras 的 fit 函数强制要求必须传 y (Label)，不传会报错。所以这里传个全 0 或全 1 的数组都可以，反正没人看它。


Train on 988129 samples
Epoch 1/20
988129/988129 [==============================] - 27s 28us/sample - loss: 4.7330
Epoch 2/20
988129/988129 [==============================] - 27s 27us/sample - loss: 4.3115
Epoch 3/20
988129/988129 [==============================] - 27s 27us/sample - loss: 4.3115
Epoch 3/20
988129/988129 [==============================] - 27s 28us/sample - loss: 4.1531
Epoch 4/20
988129/988129 [==============================] - 28s 28us/sample - loss: 4.0544
Epoch 5/20
988129/988129 [==============================] - 27s 27us/sample - loss: 3.9858
Epoch 6/20
988129/988129 [==============================] - 27s 27us/sample - loss: 3.9858
Epoch 6/20
988129/988129 [==============================] - 26s 27us/sample - loss: 3.9353
Epoch 7/20
988129/988129 [==============================] - 27s 27us/sample - loss: 3.8978
Epoch 8/20
988129/988129 [==============================] - 28s 28us/sample - loss: 3.8674
Epoch 9/20
988129/988129 [==============================] - 28s 28

In [21]:
# 4. Generate user features for testing and full item features for retrieval
#提取向量 (Generate Embeddings)
#这是**召回（Matching）模型与排序（Ranking）**模型最大的区别。 
# 训练结束后，我们需要把模型“拆”开，只要 User 塔和 Item 塔的输出，而不需要最后的 Loss 计算部分。

#准备预测数据
#用户侧使用测试集的用户数据
test_user_model_input = test_model_input
#物品侧使用全量数据库
all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}

In [22]:
#model 是一个完整的计算图。
#我们可以定义一个新的 user_embedding_model，它的输入和原模型一样，但输出截断在 User 塔的最后一层 (model.user_embedding)。Item 侧同理。
user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
###意义：现在我们拥有了两个独立的模型：
#输入用户信息 -> 输出 32维 User 向量。
#输入物品信息 -> 输出 32维 Item 向量。

In [23]:
#批量推断 (Batch Inference)
#这步操作叫做 “离线向量化”（Offline Vectorization），是召回模型从“训练阶段”走向“服务/预测阶段”的分水岭
#核心目的：把训练好的神经网络，变成两个可以查表的 “向量矩阵”。
#训练时 (Training): 我们是 Pair-wise 的（成对输入）。模型就像一个严厉的教官，同时看着 User 和 Item，计算它们匹不匹配，然后调整它们各自的权重。 模型结构：User Input + Item Input -> Score -> Loss
#这步推断 (Inference): 我们现在要把教官这一对“拆散”。
    #User Side: 让所有用户排好队，一个个通过 User 塔，算出他们每个人的“兴趣向量” (user_embs)。
    #Item Side: 让全库所有电影排好队，一个个通过 Item 塔，算出它们每个电影的“特征向量” (item_embs)。

# 1. 计算用户向量
# Input: 测试集里的用户特征 (test_user_model_input)
# Action: 跑一遍 User Embedding Model (DSSM 的左塔)
# Output: user_embs (Shape: [用户数, 32])
# 含义: 每一行代表一个用户，即使这个用户很复杂（有画像、有几百条历史行为），现在都被压缩成了这 32 个数字。
user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)


# 2. 计算物品向量
# Input: 全库所有电影的特征 (all_item_model_input)
# Action: 跑一遍 Item Embedding Model (DSSM 的右塔)
# Output: item_embs (Shape: [电影总数, 32])
# 含义: 每一行代表一部电影，这 32 个数字浓缩了电影的风格、题材等信息。
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)

(6040, 32)
(3706, 32)


e:\自学代码\搜推算法\deepmatch\venv\lib\site-packages\tensorflow\python\keras\engine\training.py:2455: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '


# 使用faiss进行ANN查找并评估结果

这几步是在进行全流程的实战演练：模拟一个真实的线上召回系统是如何工作的，并给它打分。

In [ ]:
#! pip install faiss-cpu

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [24]:
import numpy as np
import faiss
from tqdm import tqdm
from deepmatch.utils import recall_N

In [25]:
#1. 建库 (Indexing) —— "把图书馆建好"
#第一步：构建向量索引 (Build Index with Faiss)
#这一步相当于把所有商品都存入了向量数据库中，准备等待用户的查询。

# 1. 创建索引
# IndexFlatIP: "Flat"表示暴力搜索（不压缩，保证精度），"IP"代表 Inner Product（内积）。
# 在深度学习中，内积通常用于衡量向量相似度（类似于余弦相似度，如果向量归一化过的话）。
index = faiss.IndexFlatIP(embedding_dim)

## 2. 将物品向量加入索引
# item_embs: 之前算好的所有电影的 Embedding 矩阵，形状是 (电影总数, 32)。
# faiss.normalize_L2(item_embs)
index.add(item_embs)

In [26]:
#2. 检索 (Retrieval) —— "读者来借书"
#第二步：进行批量检索 (Batch Search)
#这里直接用所有测试集用户的向量去数据库里“搜”他们可能喜欢的电影。

# faiss.normalize_L2(user_embs)
## D: Distance (这里是内积得分)，形状 (用户数, 100)
# I: Index (物品的ID索引)，形状 (用户数, 100)
# search(user_embs, 100): 对每个用户向量，找出内积最大的 Top 100 个物品。
D, I = index.search(np.ascontiguousarray(user_embs), 100) # 将搜索结果从50改为100，以便后续计算Recall@100
#np.ascontiguousarray：Faiss 要求输入数据在内存中是连续的 C-order 数组，这是一个常见的工程细节。
#结果：对于第 i 个用户，I[i] 保存了推荐给他的 100 个电影在 item_profile 中的行号（索引）。

#注意“计算内积这件事，被外包给了 Faiss”
#index.search 内部在做什么？ 它在做疯狂的矩阵乘法（Matrix Multiplication）。
#数学上Scores=UserMatrix×ItemMatrix^T
#user_embs 是 U×K（用户数 × 32）
#item_embs 是 I×K（物品数 × 32）
#它们相乘得到 U×I 的分数矩阵。
#Faiss 的厉害之处： 如果让你写 for 循环去算，或者用普通的 numpy 算，面对百万级商品可能会很慢。 Faiss 用了极度优化的 C++ 代码、SIMD 指令集甚至 GPU 加速，在几毫秒内就能算完这些内积，并且顺便帮你排个序，只把最大的 100 个分数值（D）和对应的物品编号（I）吐给你。

In [29]:
D

array([[0.5030044 , 0.48093268, 0.46101317, ..., 0.3055786 , 0.30446455,
        0.30379763],
       [0.55936044, 0.5293887 , 0.52400726, ..., 0.4065881 , 0.4059934 ,
        0.40522677],
       [0.6390761 , 0.6325694 , 0.62814134, ..., 0.48598102, 0.48432323,
        0.48358393],
       ...,
       [0.26168245, 0.2421192 , 0.22563207, ..., 0.18797356, 0.18792018,
        0.1879016 ],
       [0.4776782 , 0.46691364, 0.4663713 , ..., 0.36394083, 0.3636343 ,
        0.36356264],
       [0.4276092 , 0.41133505, 0.40982443, ..., 0.30403745, 0.30401075,
        0.3038764 ]], dtype=float32)

In [30]:
I

array([[2812, 2504, 2502, ..., 1434,  208,  210],
       [ 214,   39,  860, ...,   84,  619,  248],
       [ 136,  830,  339, ...,   96,  203,   22],
       ...,
       [ 639,  533,  561, ...,   58,  573, 1199],
       [ 132,   48,   97, ...,  223,  373,  350],
       [ 160,  630, 1012, ...,  396,  229,  391]], dtype=int64)

In [27]:
#3. 评估 (Evaluation) —— "看看推得准不准"
#第三步：评估效果 (Evaluation)

# 1. 准备真实标签字典
# key是用户ID，value是该用户在测试集中实际看过的电影ID列表 (这里每个用户只有1个测试样本)
test_true_label = {line[0]:[line[1]] for line in test_set}

s = []
hit = 0


In [28]:
# 2. 遍历每个用户进行对比
# 这里一定要注意：上一步 index.search(..., 100) 必须把 k 改为 100，否则下面的 loop 拿到 pred 只有 50 个元素，算不出 K=100 的指标

# 定义存储结果的容器
metrics = {
    50: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []},
    100: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []}
}

for i, uid in tqdm(enumerate(test_user_model_input['user_id'])):
    try:
        # I[i]得到的是所有的索引(0, 1, 2...)的列表，长度是我们在 search 时指定的 (这里假设改成了 100)
        # 我们需要把它还原成真实的 movie_id
        pred_full = [item_profile['movie_id'].values[x] for x in I[i]]
        
        # 获取用户真实看过的电影列表 (在这个数据集中，长度通常为 1)
        true_items = test_true_label[uid] 

        # 分别计算 @50 和 @100 的指标
        for K in [50, 100]:
            # 截取前 K 个推荐结果
            pred_k = pred_full[:K]
            
            # --- 1. 基础命中统计 (Intersection) ---
            # 预测对了几个？
            hits = set(pred_k) & set(true_items)
            hit_count = len(hits)

            # --- 2. Hit Rate ---
            if hit_count > 0:
                metrics[K]['hit'] += 1
            
            # --- 3. Recall (召回率) ---
            # 命中数 / 真实感兴趣的总数
            # 如果真实只看了 1 个，命中就是 1/1=1，不命中就是 0/1=0
            recall_val = hit_count / len(true_items)
            metrics[K]['recall'].append(recall_val)
            
            # --- 4. Precision (精确率) ---
            # 命中数 / 推荐列表长度 K
            # 推荐了 50 个，中了 1 个，精度就是 1/50 = 2%
            precision_val = hit_count / K
            metrics[K]['precision'].append(precision_val)
            
            # --- 5. F1-Score ---
            # 调和平均数
            if (precision_val + recall_val) > 0:
                f1_val = 2 * (precision_val * recall_val) / (precision_val + recall_val)
            else:
                f1_val = 0
            metrics[K]['f1'].append(f1_val)

            # --- 6. NDCG (归一化折损累计增益) ---
            # 位置越靠前，价值越大
            
            # (1) 计算 DCG (实际增益)
            dcg_k = 0
            for rank, item in enumerate(pred_k):
                if item in true_items:
                    # rank 是从 0 开始的 (0, 1, 2...)
                    # 公式通常是 log2(rank + 2)，即第一名是 log2(2)=1，第二名是 log2(3)=1.58...
                    dcg_k += 1.0 / np.log2(rank + 2)
            
            # (2) 计算 IDCG (理想最大增益)
            # 假设所有真实感兴趣的物品都排在最前面
            idcg_k = 0
            # 理想情况下，前 len(true_items) 个位置都应该是对的
            for rank in range(min(len(true_items), K)):
                idcg_k += 1.0 / np.log2(rank + 2)
            
            # (3) 归一化
            if idcg_k > 0:
                metrics[K]['ndcg'].append(dcg_k / idcg_k)
            else:
                metrics[K]['ndcg'].append(0)

    except Exception as e:
        print(f"Error at index {i}: {e}") # 打印出错信息

# 打印最终报告
print("-" * 30)
for K in [50, 100]:
    print(f"Metrics @ {K}:")
    print(f"  Recall   : {np.mean(metrics[K]['recall']):.4f}")
    print(f"  Precision: {np.mean(metrics[K]['precision']):.4f}")
    print(f"  NDCG     : {np.mean(metrics[K]['ndcg']):.4f}")
    print(f"  F1-Score : {np.mean(metrics[K]['f1']):.4f}")
    print(f"  Hit Rate : {metrics[K]['hit'] / len(test_user_model_input['user_id']):.4f}")
    print("-" * 30)

0it [00:00, ?it/s]

6040it [00:00, 6938.08it/s]

------------------------------
Metrics @ 50:
  Recall   : 0.3581
  Precision: 0.0072
  NDCG     : 0.1113
  F1-Score : 0.0140
  Hit Rate : 0.3581
------------------------------
Metrics @ 100:
  Recall   : 0.4892
  Precision: 0.0049
  NDCG     : 0.1326
  F1-Score : 0.0097
  Hit Rate : 0.4892
------------------------------
